### 1. Loading Data

In [0]:
%fs ls '/Volumes/cat_spark_streaming/landing/datasets'

In [0]:
data_path='/Volumes/cat_spark_streaming/landing/datasets'
catalog='cat_spark_streaming'
schema='wcnt'

In [0]:
lines=spark.read \
    .format('text') \
    .option('multiLine', True) \
    .option('lineSep', '.') \
    .load(data_path)

## 2. Processing Data

In [0]:
from pyspark.sql.functions import explode,split,trim,lower,col,count,substring
from pyspark.sql.functions import *

Trim the spaces from both ends for the specified string column

In [0]:
raw_words=lines.select(explode(split(col('value')," ")).alias('word'))

quality_words=raw_words.select(trim(col('word')).alias('word') )

quality_words=quality_words.select((split(col('word'),',')[0]).alias('word'))

quality_words=quality_words.select(col('word')) \
                        .where("word is not null") \
                        .where("word rlike '[a-z]'") \
                


In [0]:
quality_words=quality_words.select(col('word')) \
                        .where("word is not null") \
                        .where("word rlike '[a-z]'") 

In [0]:
quality_words.display()

In [0]:
WordCounts=quality_words \
        .groupBy('word') \
        .agg(count(col('word')).alias('wordCount'))

In [0]:
WordCounts.show()

In [0]:
actual_count = spark.sql(f"select sum(wordCount) from {catalog}.{schema}.wordCountBatch where substr(word, 1, 1) == 's'").collect()[0][0]


In [0]:
actual_count

In [0]:
WordCounts.write \
            .mode('overwrite') \
            .format('delta') \
            .saveAsTable(f'{catalog}.{schema}.wordCountBatch') 